# Data Analysis Graph

This notebook implements a LangGraph workflow for data analysis using AI agents.

## Setup and Configuration

In [ ]:
# Import required libraries
import sys
from pathlib import Path

# Add project root to path
project_root = Path().absolute().parent.parent
sys.path.insert(0, str(project_root))

# Import settings
from src.config import settings

print(f"Application: {settings.app_name}")
print(f"Version: {settings.app_version}")
print(f"Default LLM Model: {settings.default_llm_model}")
print(f"Temperature: {settings.default_llm_temperature}")


Application: Data Agent LangGraph
Version: 0.1.0
Default LLM Model: claude-3-5-sonnet-20241022
Temperature: 0.7


## Import Dependencies

In [ ]:
from typing import TypedDict, Annotated
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langchain_anthropic import ChatAnthropic
from langchain_core.messages import HumanMessage, AIMessage
import pandas as pd
import numpy as np

## Define Graph State

In [ ]:
class DataAnalysisState(TypedDict):
    """State for the data analysis graph."""
    messages: Annotated[list, add_messages]
    data: pd.DataFrame
    analysis_results: dict

## Initialize LLM

In [ ]:
# Initialize the LLM using settings
llm = ChatAnthropic(
    model=settings.default_llm_model,
    temperature=settings.default_llm_temperature,
    api_key=settings.anthropic_api_key
)

print(f"LLM initialized: {settings.default_llm_model}")

## Define Graph Nodes

In [ ]:
def load_data_node(state: DataAnalysisState) -> DataAnalysisState:
    """Load sample data for analysis."""
    # Create sample data
    data = pd.DataFrame({
        'product': ['A', 'B', 'C', 'D', 'E'],
        'sales': [100, 150, 200, 175, 225],
        'revenue': [1000, 2250, 4000, 3500, 5625]
    })
    
    state['data'] = data
    state['messages'].append(AIMessage(content="Data loaded successfully"))
    return state


def analyze_data_node(state: DataAnalysisState) -> DataAnalysisState:
    """Analyze the data using AI."""
    data = state['data']
    
    # Prepare data summary for LLM
    summary = f"""
    Dataset Summary:
    - Shape: {data.shape}
    - Columns: {list(data.columns)}
    - Total Sales: {data['sales'].sum()}
    - Total Revenue: {data['revenue'].sum()}
    - Average Sales: {data['sales'].mean():.2f}
    - Average Revenue: {data['revenue'].mean():.2f}
    """
    
    # Get AI analysis
    response = llm.invoke([
        HumanMessage(content=f"Analyze this sales data and provide insights:\n{summary}")
    ])
    
    state['analysis_results'] = {
        'summary': summary,
        'ai_insights': response.content
    }
    state['messages'].append(AIMessage(content="Analysis complete"))
    
    return state

## Build the Graph

In [ ]:
# Create the graph
workflow = StateGraph(DataAnalysisState)

# Add nodes
workflow.add_node("load_data", load_data_node)
workflow.add_node("analyze_data", analyze_data_node)

# Add edges
workflow.add_edge(START, "load_data")
workflow.add_edge("load_data", "analyze_data")
workflow.add_edge("analyze_data", END)

# Compile the graph
graph = workflow.compile()

print("Graph compiled successfully!")

## Run the Graph

In [ ]:
# Initialize state
initial_state = {
    "messages": [HumanMessage(content="Start data analysis")],
    "data": pd.DataFrame(),
    "analysis_results": {}
}

# Run the graph
result = graph.invoke(initial_state)

print("\n" + "="*50)
print("EXECUTION COMPLETE")
print("="*50)

## Display Results

In [ ]:
# Display the data
print("\nData:")
display(result['data'])

# Display analysis results
print("\n" + "="*50)
print("ANALYSIS RESULTS")
print("="*50)
print(result['analysis_results']['summary'])
print("\nAI Insights:")
print(result['analysis_results']['ai_insights'])

## Visualize the Graph (Optional)

In [ ]:
# Uncomment if you have graphviz installed
# from IPython.display import Image, display
# display(Image(graph.get_graph().draw_mermaid_png()))